# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_two.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_two.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.999980,0.000019,7.173664e-07,0.999940,0.000059,0.000001,0.999902,0.000087,0.000011,0.999964,0.000035,9.391767e-07,9.999364e-01,0.000064,0.000000,0.998533,0.001276,0.000191
1,0.993270,0.000283,6.446509e-03,0.994368,0.000245,0.005387,0.993982,0.000245,0.005772,0.992230,0.000265,7.504966e-03,9.951892e-01,0.000171,0.004639,0.972023,0.002391,0.025586
2,0.000034,0.999962,4.381048e-06,0.000139,0.999859,0.000002,0.000214,0.999774,0.000012,0.000013,0.999981,6.276192e-06,3.944814e-07,1.000000,0.000000,0.000114,0.999862,0.000024
3,0.999879,0.000120,7.617780e-07,0.999812,0.000187,0.000001,0.999843,0.000145,0.000012,0.999931,0.000069,5.789762e-07,9.998199e-01,0.000180,0.000000,0.998349,0.001455,0.000196
4,0.997913,0.002069,1.828613e-05,0.998790,0.001202,0.000008,0.998263,0.001704,0.000033,0.996330,0.003666,3.709336e-06,9.977873e-01,0.002205,0.000008,0.994559,0.004657,0.000784


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.998106,0.001851,0.000042,0.997287,0.002522,0.000192,0.998122,0.001845,0.000033,0.993568,0.006380,0.000053,0.998112,0.001858,0.000030,0.991100,0.006021,0.002879
1,0.997635,0.002352,0.000014,0.997773,0.002224,0.000003,0.997897,0.002071,0.000032,0.996850,0.003147,0.000003,0.998374,0.001623,0.000003,0.993814,0.005762,0.000424
2,0.997136,0.001323,0.001541,0.997897,0.000212,0.001892,0.997850,0.000528,0.001622,0.992689,0.000960,0.006351,0.997662,0.000721,0.001618,0.987143,0.003596,0.009262
3,0.000485,0.000066,0.999449,0.001142,0.000192,0.998666,0.000653,0.000081,0.999266,0.000062,0.000010,0.999928,0.000413,0.000015,0.999572,0.000552,0.000322,0.999126
4,0.999485,0.000510,0.000004,0.999636,0.000360,0.000003,0.999666,0.000319,0.000016,0.999455,0.000531,0.000014,0.999776,0.000222,0.000002,0.998133,0.001603,0.000264


# Machine Learning

In [6]:
models = dict(
    lgbm=load_pickle('../models/layer_3/model_lightgbm.pkl'),
    cat=load_pickle('../models/layer_3/model_catboost.pkl'),
    xgb=load_pickle('../models/layer_3/model_xgboost.pkl'),
    hist=load_pickle('../models/layer_3/model_hist_gradient_boosting.pkl'),
    rf=load_pickle('../models/layer_3/model_random_forest.pkl'),
    extra=load_pickle('../models/layer_3/model_extra_tree.pkl'),
)

## Train Dataset

In [7]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

In [8]:
X_train_stacking = pd.DataFrame({})

In [9]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(model, X_train, y_train.class_encoded, cv=cv, n_jobs=-1, method='predict_proba')
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                         | 0/6 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 17%|█████████████████████████████▎                                                                                                                                                  | 1/6 [03:11<15:58, 191.80s/it]

Predicting Train Dataset cat


 33%|██████████████████████████████████████████████████████████▋                                                                                                                     | 2/6 [11:15<24:14, 363.55s/it]

...

 67%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                          | 4/6 [13:04<05:02, 151.11s/it]

Predicting Train Dataset rf


 83%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                             | 5/6 [24:39<05:47, 347.05s/it]

Predicting Train Dataset extra


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [25:52<00:00, 258.80s/it]


## Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({})

In [11]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

  0%|                                                                                                                                                                                         | 0/6 [00:00<?, ?it/s]

Predicting Test Dataset lgbm


 17%|█████████████████████████████▌                                                                                                                                                   | 1/6 [00:32<02:43, 32.78s/it]

Predicting Test Dataset cat


 33%|███████████████████████████████████████████████████████████                                                                                                                      | 2/6 [00:33<00:54, 13.64s/it]

...

 67%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                                                           | 4/6 [00:33<00:09,  4.79s/it]

Predicting Test Dataset rf


 83%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                             | 5/6 [00:35<00:03,  3.61s/it]

Predicting Test Dataset extra


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:37<00:00,  6.18s/it]


# Saving

In [12]:
X_train_stacking.to_parquet('../data/X_train_stacking_layer_three.parquet')
X_test_stacking.to_parquet('../data/X_test_stacking_layer_three.parquet')

In [13]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.999778,0.000212,0.000009,0.999787,0.000174,0.000040,0.999864,0.000121,0.000016,0.999733,0.000264,2.459617e-06,0.999775,0.000225,2.616556e-08,0.998912,0.000955,0.000133
1,0.994627,0.000176,0.005197,0.995247,0.000365,0.004388,0.992235,0.000199,0.007566,0.973059,0.000137,2.680453e-02,0.994737,0.000047,5.216709e-03,0.976808,0.000850,0.022342
2,0.000048,0.999944,0.000008,0.000483,0.999485,0.000032,0.000169,0.999822,0.000009,0.000015,0.999984,1.319155e-06,0.000011,0.999989,9.836632e-08,0.000068,0.999897,0.000035
3,0.999915,0.000081,0.000005,0.999745,0.000217,0.000039,0.999858,0.000127,0.000014,0.999960,0.000040,6.602657e-07,0.999793,0.000207,3.828407e-08,0.998919,0.000947,0.000134
4,0.998164,0.001812,0.000024,0.998143,0.001764,0.000093,0.998504,0.001459,0.000037,0.986426,0.013556,1.806930e-05,0.998227,0.001769,3.566911e-06,0.994430,0.005248,0.000322


In [14]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.998177,0.001784,0.000039,0.997654,0.002153,0.000193,0.998108,0.001843,0.000049,0.997980,0.002008,0.000012,0.998183,0.001800,0.000017,0.993598,0.005832,0.000569
1,0.997269,0.002710,0.000021,0.997945,0.001978,0.000077,0.997185,0.002778,0.000037,0.994660,0.005324,0.000015,0.997905,0.002092,0.000003,0.993669,0.005970,0.000361
2,0.996089,0.000609,0.003303,0.997698,0.000711,0.001592,0.997502,0.000382,0.002115,0.997295,0.001499,0.001207,0.997506,0.000715,0.001780,0.988872,0.002584,0.008544
3,0.000643,0.000107,0.999250,0.001658,0.000165,0.998177,0.000730,0.000132,0.999138,0.000139,0.000020,0.999841,0.000541,0.000023,0.999436,0.000378,0.000214,0.999409
4,0.999631,0.000357,0.000012,0.999578,0.000365,0.000058,0.999737,0.000245,0.000018,0.999526,0.000469,0.000005,0.999785,0.000205,0.000010,0.998670,0.001116,0.000214
